# Pretraining notebook

## Section 1: Generate text before training

Load your current model

Enter a few prompts and note the output.

Try these:
 - love is
 - baby
 - tonight

Generate 20 tokens and note the output.

Reflection: Is there any quality or meaning to the output or is it largely random text?  Why?

In [58]:
import sys
import torch
torch.manual_seed(123)

from weird_ai.config import SAMPLE_LYRICS_FILE, TOKENS_FILE
from weird_ai.model import WeirdAIModel
from weird_ai.tokenizer import SimpleCharacterTokenizer
from weird_ai.generation import (
    text_to_token_ids, token_ids_to_text, generate_text_simple,
)

device = torch.device("cuda")

corpus = SAMPLE_LYRICS_FILE.read_text(encoding="utf-8")
tokenizer = SimpleCharacterTokenizer(corpus)
vocab_size = len(tokenizer.chars)
print("vocab_size:", vocab_size)

# giving the model its puny context size
context_size = 128
model = WeirdAIModel(vocab_size).to(device)

# prompt generation
for prompt in ["love is", "baby", "tonight"]:
    ids = text_to_token_ids(prompt, tokenizer).to(device)
    out = generate_text_simple(model, ids, max_new_tokens=20,
                               context_size=context_size)
    print(f"\nPrompt: {prompt!r}")
    print(token_ids_to_text(out, tokenizer))


vocab_size: 681

Prompt: 'love is'
love is서c배âr표음뻐셔완놀서<설ミ년á년가요

Prompt: 'baby'
baby入⁠打서c뜻모모블飛色圖﻿잊말~난í털Д

Prompt: 'tonight'
tonight日Ë뒤ï방l음S얼또떠’#란ミ裡아따줄시


## Section 2: Understanding Logits

Using the logits starter code, do the following:
 - Apply softmax
 - Identify the highest probability token

Reflection: Why is softmax necessary before interpreting logits as probabilities?

In [16]:
logits = torch.tensor([
    [1.5, 2.0, 0.5]
])

probabilities = torch.softmax(logits, dim=-1)

print(probabilities)
print(probabilities.sum(dim=-1))

predicted_token = torch.argmax(probabilities, dim=-1)

print(predicted_token)

tensor([[0.3315, 0.5465, 0.1220]])
tensor([1.0000])
tensor([1])


## Section 3: Cross Entropy Loss

### Textbook 
The textbook discussion begins on page 136

### Manually calculate
 - Probabilities
 - Log probabilities
 - Average negative log probabilities

### Compare with torch
Compare your manual results to the torch cross entropy results

```python
     torch.nn.functional.cross_entropy(...)
'''


In [17]:
import torch
import torch.nn.functional as F

probs = torch.tensor([0.7, 0.2, 0.1])
target_index = 0

In [18]:
target_probability = probs[target_index]
print(target_probability)

tensor(0.7000)


In [19]:
log_probability = torch.log(target_probability)
print(log_probability)

tensor(-0.3567)


In [20]:
loss = -log_probability
print(loss)

tensor(0.3567)


In [ ]:
logits = torch.log(probs).unsqueeze(0)
target = torch.tensor([target_index])

torch_loss = F.cross_entropy(logits, target)
print("manual loss :", loss.item())
print("torch loss  :", torch_loss.item())

manual loss : 0.3566749691963196
torch loss  : 0.3566749691963196


## Section 4: Perplexity

Compute the perplexity

```python
perplexity = torch.exp(loss)
```

**Note:** A perplexity of 10 means the model is roughly as uncertain as choosing among 10 equally likely next tokens.

In [22]:
loss = torch.tensor(2.5)

perplexity = torch.exp(loss)

print(perplexity)

tensor(12.1825)


## Section 5: Understanding the Training Loop

The training loop performs:

1. Iterate through epochs
2. Iterate through batches
3. Zero gradients
4. Calculate loss
5. Backpropagation
6. Optimizer step
7. Evaluate model
8. Generate sample text

Draw or explain this process in your own words.

# Training vs. Validation

You will:
 - Split lyrics into train/validation
 - Create loaders
 - Compute initial loss

Keep note of your: 
 - Training Loss: 
 - Validation Loss: 

In [23]:
from weird_ai.config import TOKENS_FILE
from weird_ai.dataset import LyricsDataset
from torch.utils.data import DataLoader

tokens = torch.tensor(
    [int(x) for x in TOKENS_FILE.read_text(encoding="utf-8").split()],
    dtype=torch.long,
)
print("total tokens:", tokens.numel())

train_ratio = 0.9
split = int(len(tokens) * train_ratio)

train_tokens = tokens[:split]
val_tokens   = tokens[split:]

print("Training tokens:  ", len(train_tokens))
print("Validation tokens:", len(val_tokens))

total tokens: 13507738
Training tokens:   12156964
Validation tokens: 1350774


In [ ]:
block_size = 64
batch_size = 1024 # pat myself on the back for big VRAM

train_dataset = LyricsDataset(train_tokens.tolist(), block_size)
val_dataset   = LyricsDataset(val_tokens.tolist(),   block_size)

train_loader = DataLoader(
    train_dataset, 
    batch_size=batch_size, 
    shuffle=True, 
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,   
    batch_size=batch_size, 
    shuffle=False, 
    num_workers=4, 
    pin_memory=True
)

print(train_loader)
print(val_loader)

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss()

# decided to cut it off early because it was going over the ENTIRE training data and that's probably not necessary to calculate the avgs
def evaluate_loss(model, loader, tag="", max_batches=100):
    model.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for i, (x, y) in enumerate(loader):
            if i >= max_batches:
                break
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = loss_fn(logits.view(-1, logits.size(-1)), y.view(-1))
            total += loss.item()
            n += 1
    return total / max(n, 1)

initial_train_loss = evaluate_loss(model, train_loader, "train")
initial_val_loss   = evaluate_loss(model, val_loader,   "val")

print(f"Initial training loss:   {initial_train_loss:.4f}")
print(f"Initial validation loss: {initial_val_loss:.4f}")

Initial training loss:   6.7119
Initial validation loss: 6.7126


## Section 6: Training

Train for one Epoch
```python
train_model(...)
```

Then record the loss before and after training.  

In [ ]:
from tqdm.auto import tqdm

num_epochs = 1
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
loss_fn = torch.nn.CrossEntropyLoss()

def train_one_epoch(model, loader, optimizer, loss_fn):
    model.train()
    total_loss = 0.0

    # finally saw a reasonable place to put this sorta thing
    progress_bar = tqdm(loader, desc="Training", leave=False)
    
    for x, y in progress_bar:
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()
        logits = model(x)
        loss = loss_fn(logits.view(-1, logits.size(-1)), y.view(-1))
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})
        
    return total_loss / len(loader)

def evaluate_loss(model, loader):
    model.eval()
    total_loss = 0.0
    progress_bar = tqdm(loader, desc="Validating", leave=False)
    
    with torch.no_grad():
        for x, y in progress_bar:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = loss_fn(logits.view(-1, logits.size(-1)), y.view(-1))
            total_loss += loss.item()
            progress_bar.set_postfix({"loss": f"{loss.item():.4f}"})
            
    return total_loss / len(loader)

train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn)
val_loss = evaluate_loss(model, val_loader)
print(f"\nEpoch complete; train loss: {train_loss:.4f}. Val loss: {val_loss:.4f}")


Epoch complete; train loss: 1.8412. Val loss: 1.5616


In [67]:
print(f"Initial loss: {initial_train_loss:.4f}")
print(f"Final loss:   {train_loss:.4f}")
print(f"Delta loss:       {initial_train_loss - train_loss:+.4f}")

prompt = "you have"
ids = text_to_token_ids(prompt, tokenizer).to(device)
out = generate_text_simple(model, ids, max_new_tokens=40,
                               context_size=context_size)
print(f"\nPrompt: {prompt!r}")
print(token_ids_to_text(out, tokenizer))

Initial loss: 6.7119
Final loss:   1.8412
Delta loss:       +4.8707

Prompt: 'you have'
you have me the started to the start
I want the 


In [ ]:
# building lists for saving ckpt
train_losses = [train_loss]
val_losses = [val_loss]
tokens_seen = len(train_loader) * train_loader.batch_size * train_loader.dataset.block_size
track_tokens_seen = [tokens_seen]

## Evaluation

Did the loss decrease?

Why is decreasing loss important?

In [71]:
# actual save
from pathlib import Path
from weird_ai.config import PROJECT_ROOT
from weird_ai.trainer import save_checkpoint

ckpt_dir = PROJECT_ROOT / "models" / "lesson-05-pretrained"
ckpt_dir.mkdir(parents=True, exist_ok=True)
save_checkpoint(
    model=model, optimizer=optimizer, epoch=1,
    train_losses=train_losses, val_losses=val_losses,
    track_tokens_seen=track_tokens_seen,
    checkpoint_path=str(ckpt_dir / "checkpoint.pt"),
)
print("Saved")

Saved


## Before and After Comparison

Prompt:

love is

Before Training:
__________________

After Training:
__________________

Reflection:

How did the output change?

What evidence do you see that the model learned something from the training data?